# LangGraph Agent
## Without tools

## Library Imports

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph, MessagesState
import logging
from IPython.display import Image, Markdown, display
from langchain_core.runnables.graph import CurveStyle, MermaidDrawMethod, NodeStyles

## Configure logging

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig (
    filename='langgraph_notebook.log',
    level=logging.INFO
)

## Configure Agent and Model

In [ ]:
model = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

def call_model(state: MessagesState):
    messages = state['messages']
    response = model.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState)

workflow.add_node("agent", call_model)

workflow.add_edge(START, "agent")

checkpointer = MemorySaver()

app = workflow.compile(checkpointer=checkpointer)

## Visualize the Graph

In [ ]:
display(
    Image(
        app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

## Chat with the agent

In [ ]:
final_state = app.invoke(
    {"messages": [HumanMessage(content="What teams did Joe Montana play for?")]},
    config={"configurable": {"thread_id": 99}}
)
display(Markdown(final_state["messages"][-1].content))

In [ ]:
final_state = app.invoke(
    {"messages": [HumanMessage(content="What are the leagues in the SportsWorkdCentral fantasy fottball platform")]},
    config={"configurable": {"thread_id": 99}}
)
display(Markdown(final_state["messages"][-1].content))

In [ ]:
print("HUMAN MESSAGE:")
print(final_state["messages"][0].content)
print("\nAI MESSAGE:")
print(final_state["messages"][1].content)

## Test code to find out which model works

In [ ]:
import os 
print(os.environ.get("ANTHROPIC_API_KEY"))

In [ ]:
from anthropic import Anthropic

try:
    client = Anthropic()
    # This will fail immediately if key is wrong
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print("API key works!")
    print(response.content[0].text)
except Exception as e:
    print(f"API key problem: {e}")